## Movie Recommender Comparison

In [1]:
import pandas as pd
import numpy as np

ratings = pd.read_csv("dataset/ratings.csv")  # userId, movieId, rating
movies = pd.read_csv("dataset/movies.csv")    # movieId, title, genres

print(ratings.head())
print(movies.head())

   userId  movieId  rating  timestamp
0       1        1     4.0  964982703
1       1        3     4.0  964981247
2       1        6     4.0  964982224
3       1       47     5.0  964983815
4       1       50     5.0  964982931
   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  


## Leave-One-Out

In [2]:
def leave_one_out_split(ratings_df):
    # Shuffle to avoid bias if no timestamp
    ratings_df = ratings_df.sample(frac=1, random_state=42)

    test = ratings_df.groupby("userId").head(1)
    train = ratings_df.drop(test.index)

    return train, test


train_df, test_df = leave_one_out_split(ratings)

print("Train size:", len(train_df))
print("Test size:", len(test_df))

Train size: 100226
Test size: 610


## Build popularity baseline

In [3]:
def build_popularity_baseline(train_df, min_ratings=5):
    movie_stats = (
        train_df
        .groupby("movieId")
        .agg(
            avg_rating=("rating", "mean"),
            rating_count=("rating", "count")
        )
        .reset_index()
    )

    #filter low-count
    movie_stats = movie_stats[movie_stats["rating_count"] >= min_ratings]

    #sort by rating then count
    movie_stats = movie_stats.sort_values(
        by=["avg_rating", "rating_count"],
        ascending=False
    )

    return movie_stats


popularity_df = build_popularity_baseline(train_df)

popularity_df.head(10)

,movieId,avg_rating,rating_count
4385,6460,4.900000,5
9588,177593,4.750000,8
5753,31364,4.700000,5
1662,2239,4.666667,6
1424,1949,4.600000,5
3200,4334,4.600000,5
796,1041,4.590909,11
8274,106642,4.571429,7
2576,3451,4.545455,11
881,1178,4.541667,12


In [6]:
def recommend_popular_movies(user_id, train_df, popularity_df, n=10):
    seen_movies = set(train_df[train_df["userId"] == user_id]["movieId"])

    recs = popularity_df[
        ~popularity_df["movieId"].isin(seen_movies)
    ].head(n)

    return recs["movieId"].tolist()


#example
user_id = ratings["userId"].iloc[0]
rec_ids = recommend_popular_movies(user_id, train_df, popularity_df, n=10)

print(rec_ids)

[6460, 177593, 31364, 2239, 1949, 4334, 1041, 106642, 3451, 1178]


In [7]:
def show_recommendations(movie_ids, movies_df):
    return movies_df[movies_df["movieId"].isin(movie_ids)][["movieId", "title"]]


show_recommendations(rec_ids, movies)

,movieId,title
796,1041,Secrets & Lies (1996)
883,1178,Paths of Glory (1957)
1426,1949,"Man for All Seasons, A (1966)"
1664,2239,Swept Away (Travolti da un insolito destino ne...
2582,3451,Guess Who's Coming to Dinner (1967)
3210,4334,Yi Yi (2000)
4396,6460,"Trial, The (Procès, Le) (1962)"
5773,31364,Memories of Murder (Salinui chueok) (2003)
8301,106642,"Day of the Doctor, The (2013)"
9618,177593,"Three Billboards Outside Ebbing, Missouri (2017)"


In [9]:
def precision_at_k(recommended, relevant, k):
    recommended_k = recommended[:k]
    hits = len(set(recommended_k) & set(relevant))
    return hits / k

def recall_at_k(recommended, relevant, k):
    recommended_k = recommended[:k]
    hits = len(set(recommended_k) & set(relevant))
    return hits / len(relevant) if len(relevant) > 0 else 0

def hit_rate_at_k(recommended, relevant, k):
    recommended_k = recommended[:k]
    return int(len(set(recommended_k) & set(relevant)) > 0)

def evaluate_baseline(train_df, test_df, popularity_df, k=10):
    precisions = []
    recalls = []
    hits = []

    for _, row in test_df.iterrows():
        user_id = row["userId"]
        test_movie = row["movieId"]

        recs = recommend_popular_movies(user_id, train_df, popularity_df, n=k)
        relevant = [test_movie]

        precisions.append(precision_at_k(recs, relevant, k))
        recalls.append(recall_at_k(recs, relevant, k))
        hits.append(hit_rate_at_k(recs, relevant, k))

    return {
        "Model": "Popularity Baseline",
        f"Precision@{k}": np.mean(precisions),
        f"Recall@{k}": np.mean(recalls),
        f"HitRate@{k}": np.mean(hits)
    }


results = evaluate_baseline(train_df, test_df, popularity_df, k=10)

results_df = pd.DataFrame([results])
print(results_df)

,Model,Precision@10,Recall@10,HitRate@10
0,Popularity Baseline,0.000164,0.001639,0.001639


## kNN

In [10]:
from sklearn.neighbors import NearestNeighbors

def build_user_item_matrix(train_df):
    """
    Rows = users
    Columns = movies
    Values = ratings
    Missing ratings filled with 0 for cosine-based kNN.
    """
    user_item_matrix = train_df.pivot_table(
        index="userId",
        columns="movieId",
        values="rating"
    ).fillna(0)

    return user_item_matrix


user_item_matrix = build_user_item_matrix(train_df)

print("User-item matrix shape:", user_item_matrix.shape)
user_item_matrix.head()

User-item matrix shape: (610, 9712)


movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,4.0,0.0,4.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
